In [2]:
!pip install faker

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 32.6 MB/s  0:00:00


In [1]:
import pandas as pd
import numpy as np
import random

from faker import Faker
from datetime import datetime, timedelta

In [9]:
# =========================================================
# ENTERPRISE WORKFORCE PLANNING SYNTHETIC DATA GENERATOR
# =========================================================
# Purpose:
# Generate realistic enterprise-scale workforce planning
# datasets for:
# - Workforce analytics
# - Capacity planning
# - Headcount forecasting
# - Attrition analysis
# - Hiring pipeline analytics
# - Scenario modeling
#
# Output Tables:
# 1. employee_master.csv
# 2. workforce_capacity.csv
# 3. hiring_pipeline.csv
# 4. attrition_history.csv
# 5. workforce_demand_forecast.csv
# =========================================================

fake = Faker('en_CA')
np.random.seed(42)
random.seed(42)

# =========================================================
# CONFIGURATION
# =========================================================

NUM_EMPLOYEES = 5000
START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2025, 12, 31)

DEPARTMENTS = {
    'Operations': 0.40,
    'Maintenance': 0.20,
    'Customer Service': 0.12,
    'IT': 0.08,
    'Finance': 0.06,
    'HR': 0.05,
    'Planning': 0.05,
    'Safety & Compliance': 0.04
}

ROLES = {
    'Operations': ['Transit Operator', 'Dispatch Coordinator', 'Operations Analyst'],
    'Maintenance': ['Maintenance Technician', 'Electrical Technician', 'Field Supervisor'],
    'Customer Service': ['Customer Support Rep', 'Service Coordinator'],
    'IT': ['Data Analyst', 'BI Developer', 'Systems Administrator'],
    'Finance': ['Financial Analyst', 'Payroll Specialist'],
    'HR': ['HR Analyst', 'Recruitment Specialist'],
    'Planning': ['Workforce Planner', 'Business Analyst'],
    'Safety & Compliance': ['Safety Officer', 'Compliance Specialist']
}

LOCATIONS = [
    'Toronto',
    'Mississauga',
    'Brampton',
    'Hamilton',
    'Vaughan',
    'Markham'
]

SHIFT_TYPES = ['Day', 'Night', 'Rotational']
EMPLOYMENT_TYPES = ['Full-Time', 'Contract']

# =========================================================
# SALARY RANGES
# =========================================================

SALARY_RANGES = {
    'Transit Operator': (65000, 90000),
    'Dispatch Coordinator': (70000, 95000),
    'Operations Analyst': (75000, 105000),
    'Maintenance Technician': (70000, 98000),
    'Electrical Technician': (75000, 110000),
    'Field Supervisor': (90000, 125000),
    'Customer Support Rep': (50000, 70000),
    'Service Coordinator': (60000, 80000),
    'Data Analyst': (80000, 115000),
    'BI Developer': (95000, 135000),
    'Systems Administrator': (85000, 120000),
    'Financial Analyst': (80000, 115000),
    'Payroll Specialist': (65000, 85000),
    'HR Analyst': (70000, 95000),
    'Recruitment Specialist': (70000, 95000),
    'Workforce Planner': (90000, 130000),
    'Business Analyst': (85000, 120000),
    'Safety Officer': (80000, 110000),
    'Compliance Specialist': (85000, 115000)
}

# =========================================================
# HELPER FUNCTIONS
# =========================================================


def random_date(start, end):
    delta = end - start
    random_days = random.randint(0, delta.days)
    return start + timedelta(days=random_days)



def weighted_choice(weight_dict):
    choices = list(weight_dict.keys())
    weights = list(weight_dict.values())
    return random.choices(choices, weights=weights, k=1)[0]



def get_salary(role):
    min_salary, max_salary = SALARY_RANGES[role]
    return random.randint(min_salary, max_salary)

# =========================================================
# 1. EMPLOYEE MASTER TABLE
# =========================================================

print('Generating Employee Master Table...')

employees = []

for emp_id in range(1, NUM_EMPLOYEES + 1):

    department = weighted_choice(DEPARTMENTS)
    role = random.choice(ROLES[department])
    location = random.choice(LOCATIONS)

    hire_date = random_date(datetime(2018, 1, 1), END_DATE)

    employment_type = random.choices(
        EMPLOYMENT_TYPES,
        weights=[0.88, 0.12],
        k=1
    )[0]

    shift_type = random.choices(
        SHIFT_TYPES,
        weights=[0.60, 0.15, 0.25],
        k=1
    )[0]

    salary = get_salary(role)

    overtime_risk = np.random.normal(1.0, 0.2)

    # Higher overtime pressure in Operations and Maintenance
    if department in ['Operations', 'Maintenance']:
        overtime_risk *= 1.25

    employees.append({
        'Employee_ID': f'EMP{emp_id:05d}',
        'First_Name': fake.first_name(),
        'Last_Name': fake.last_name(),
        'Department': department,
        'Role': role,
        'Location': location,
        'Hire_Date': hire_date.date(),
        'Employment_Type': employment_type,
        'Shift_Type': shift_type,
        'Salary': salary,
        'Overtime_Risk_Factor': round(overtime_risk, 2),
        'Manager_ID': f'MGR{random.randint(1, 300):04d}',
        'Status': 'Active'
    })

employee_df = pd.DataFrame(employees)

# =========================================================
# 2. ATTRITION HISTORY TABLE
# =========================================================

print('Generating Attrition History Table...')

attrition_records = []

attrition_rate = 0.14
num_attrition = int(NUM_EMPLOYEES * attrition_rate)

attrition_employees = random.sample(
    list(employee_df['Employee_ID']),
    num_attrition
)

for emp_id in attrition_employees:

    termination_date = random_date(datetime(2023, 1, 1), END_DATE)

    attrition_type = random.choices(
        ['Voluntary', 'Involuntary'],
        weights=[0.78, 0.22],
        k=1
    )[0]

    exit_reason = random.choice([
        'Resignation',
        'Retirement',
        'Career Change',
        'Performance',
        'Relocation',
        'Burnout'
    ])

    hire_date = pd.to_datetime(
        employee_df.loc[
            employee_df['Employee_ID'] == emp_id,
            'Hire_Date'
        ].values[0]
    )

    tenure_days = (termination_date - hire_date).days
    tenure_years = round(tenure_days / 365, 2)

    attrition_records.append({
        'Employee_ID': emp_id,
        'Termination_Date': termination_date.date(),
        'Attrition_Type': attrition_type,
        'Exit_Reason': exit_reason,
        'Tenure_Years': tenure_years
    })

    employee_df.loc[
        employee_df['Employee_ID'] == emp_id,
        'Status'
    ] = 'Terminated'

attrition_df = pd.DataFrame(attrition_records)


# =========================================================
# 3. WORKFORCE CAPACITY TABLE
# =========================================================

print('Generating Workforce Capacity Table...')

capacity_records = []

months = pd.date_range(
    start='2024-01-01',
    end='2025-12-01',
    freq='MS'
)

for _, employee in employee_df.iterrows():

    for month in months:

        available_hours = random.randint(150, 176)

        base_utilization = np.random.normal(0.82, 0.08)

        # Operations & Maintenance tend to have higher utilization
        if employee['Department'] in ['Operations', 'Maintenance']:
            base_utilization += 0.07

        base_utilization = min(max(base_utilization, 0.55), 1.08)

        worked_hours = int(available_hours * base_utilization)

        overtime_hours = max(
            0,
            int((worked_hours - available_hours) * employee['Overtime_Risk_Factor'])
        )

        vacation_hours = random.randint(0, 24)
        sick_hours = random.randint(0, 16)

        utilization_rate = round(
            worked_hours / available_hours,
            2
        )

        burnout_risk = 'Low'

        if utilization_rate >= 0.95:
            burnout_risk = 'High'
        elif utilization_rate >= 0.88:
            burnout_risk = 'Medium'

        capacity_records.append({
            'Employee_ID': employee['Employee_ID'],
            'Month': month.date(),
            'Department': employee['Department'],
            'Available_Hours': available_hours,
            'Worked_Hours': worked_hours,
            'Overtime_Hours': overtime_hours,
            'Vacation_Hours': vacation_hours,
            'Sick_Hours': sick_hours,
            'Utilization_Rate': utilization_rate,
            'Burnout_Risk': burnout_risk
        })

capacity_df = pd.DataFrame(capacity_records)

# =========================================================
# 4. HIRING PIPELINE TABLE
# =========================================================

print('Generating Hiring Pipeline Table...')

hiring_records = []

for req_id in range(1, 2501):

    department = weighted_choice(DEPARTMENTS)

    open_date = random_date(
        datetime(2023, 1, 1),
        END_DATE
    )

    candidate_count = random.randint(8, 150)

    # Harder-to-fill technical roles
    if department in ['IT', 'Planning']:
        time_to_fill = random.randint(45, 120)
    else:
        time_to_fill = random.randint(18, 75)

    hire_date = open_date + timedelta(days=time_to_fill)

    offer_accepted = random.choices(
        ['Yes', 'No'],
        weights=[0.83, 0.17],
        k=1
    )[0]

    hiring_records.append({
        'Requisition_ID': f'REQ{req_id:05d}',
        'Department': department,
        'Open_Date': open_date.date(),
        'Time_To_Fill_Days': time_to_fill,
        'Candidate_Count': candidate_count,
        'Offer_Accepted': offer_accepted,
        'Hire_Date': hire_date.date(),
        'Hiring_Manager': f'MGR{random.randint(1, 300):04d}'
    })

hiring_df = pd.DataFrame(hiring_records)

# =========================================================
# 5. WORKFORCE DEMAND FORECAST TABLE
# =========================================================

print('Generating Workforce Demand Forecast Table...')

forecast_records = []

forecast_months = pd.date_range(
    start='2024-01-01',
    end='2026-12-01',
    freq='MS'
)

base_headcount = NUM_EMPLOYEES

for month_num, month in enumerate(forecast_months):

    seasonal_factor = 1 + (0.05 * np.sin(month_num / 2))

    growth_factor = 1 + (month_num * 0.008)

    operational_demand = int(
        10000 * seasonal_factor * growth_factor
    )

    required_headcount = int(
        base_headcount * seasonal_factor * growth_factor
    )

    attrition_impact = random.randint(20, 120)

    actual_headcount = required_headcount - attrition_impact

    workforce_gap = required_headcount - actual_headcount

    overtime_dependency = round(
        workforce_gap / required_headcount,
        3
    )

    hiring_need = max(0, workforce_gap + random.randint(5, 30))

    forecast_records.append({
        'Month': month.date(),
        'Forecasted_Demand': operational_demand,
        'Required_Headcount': required_headcount,
        'Actual_Headcount': actual_headcount,
        'Workforce_Gap': workforce_gap,
        'Projected_Hiring_Need': hiring_need,
        'Overtime_Dependency_Rate': overtime_dependency
    })

forecast_df = pd.DataFrame(forecast_records)


# =========================================================
# EXPORT FILES
# =========================================================

print('Exporting CSV files...')

employee_df.to_csv('employee_master.csv', index=False)
capacity_df.to_csv('workforce_capacity.csv', index=False)
hiring_df.to_csv('hiring_pipeline.csv', index=False)
attrition_df.to_csv('attrition_history.csv', index=False)
forecast_df.to_csv('workforce_demand_forecast.csv', index=False)

# =========================================================
# DATA QUALITY SUMMARY
# =========================================================

print('\n=========================================')
print('DATA GENERATION COMPLETE')
print('=========================================')
print(f'Employee Records: {len(employee_df):,}')
print(f'Capacity Records: {len(capacity_df):,}')
print(f'Hiring Records: {len(hiring_df):,}')
print(f'Attrition Records: {len(attrition_df):,}')
print(f'Forecast Records: {len(forecast_df):,}')
print('=========================================')

# =========================================================
# OPTIONAL: SAMPLE KPI OUTPUTS
# =========================================================

headcount = employee_df['Employee_ID'].nunique()
active_employees = employee_df[
    employee_df['Status'] == 'Active'
]['Employee_ID'].nunique()

attrition_percentage = round(
    len(attrition_df) / headcount * 100,
    2
)

avg_utilization = round(
    capacity_df['Utilization_Rate'].mean() * 100,
    2
)

avg_time_to_fill = round(
    hiring_df['Time_To_Fill_Days'].mean(),
    1
)

print('\nENTERPRISE WORKFORCE KPI SUMMARY')
print('-----------------------------------------')
print(f'Total Headcount: {headcount:,}')
print(f'Active Employees: {active_employees:,}')
print(f'Attrition Rate: {attrition_percentage}%')
print(f'Average Utilization: {avg_utilization}%')
print(f'Average Time To Fill: {avg_time_to_fill} days')
print('-----------------------------------------')


Generating Employee Master Table...
Generating Attrition History Table...
Generating Workforce Capacity Table...
Generating Hiring Pipeline Table...
Generating Workforce Demand Forecast Table...
Exporting CSV files...

DATA GENERATION COMPLETE
Employee Records: 5,000
Capacity Records: 120,000
Hiring Records: 2,500
Attrition Records: 700
Forecast Records: 36

ENTERPRISE WORKFORCE KPI SUMMARY
-----------------------------------------
Total Headcount: 5,000
Active Employees: 4,300
Attrition Rate: 14.0%
Average Utilization: 85.9%
Average Time To Fill: 51.5 days
-----------------------------------------
